In [ ]:
#@title 🚀 CELDA 1 — Levantar servidor (Uvicorn + ngrok Free)
import os
import sys
import time
import subprocess
import requests
import nest_asyncio
from google.colab import drive
from pyngrok import ngrok, conf

BASE_DIR = '/content/drive/MyDrive/BluegridOCRv3'
API_DIR  = os.path.join(BASE_DIR, 'backend_api')
HOST = '127.0.0.1'
PORT = 8000

# ── Tomar credenciales desde variables ya definidas o entorno ────────────────
ANTHROPIC_API_KEY = globals().get('ANTHROPIC_API_KEY', os.environ.get('ANTHROPIC_API_KEY', ''))
NGROK_TOKEN       = globals().get('NGROK_TOKEN', os.environ.get('NGROK_TOKEN', ''))

# ── Montar Drive si no está montado ───────────────────────────────────────────
if not os.path.exists('/content/drive/MyDrive'):
    print('🔌 Montando Drive...')
    drive.mount('/content/drive', force_remount=True)
else:
    print('✅ Drive ya montado.')

# ── Validación básica de credenciales ─────────────────────────────────────────
faltantes = []
if not ANTHROPIC_API_KEY:
    faltantes.append('ANTHROPIC_API_KEY')
if not NGROK_TOKEN:
    faltantes.append('NGROK_TOKEN')

if faltantes:
    raise SystemExit(f'❌ Faltan variables requeridas: {", ".join(faltantes)}')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

# ── Diagnóstico ───────────────────────────────────────────────────────────────
print('\n🔍 Diagnóstico pre-arranque:')
criticos = [
    os.path.join(API_DIR, 'main.py'),
    os.path.join(API_DIR, 'services', 'db.py'),
    os.path.join(API_DIR, 'services', 'motor_ia.py'),
    os.path.join(API_DIR, 'routers', 'operations.py'),
    os.path.join(API_DIR, 'routers', 'auth.py'),
    os.path.join(API_DIR, 'routers', 'dashboard.py'),
]

todo_ok = True
for f in criticos:
    existe = os.path.exists(f)
    print(f'  {"✅" if existe else "❌ FALTA"}  {os.path.relpath(f, BASE_DIR)}')
    if not existe:
        todo_ok = False

if not todo_ok:
    raise SystemExit('❌ Ejecuta la celda que crea/restaura backend_api primero.')

print('✅ ANTHROPIC_API_KEY configurada.' if ANTHROPIC_API_KEY.startswith('sk-ant-') else '⚠️ API Key parece no tener prefijo esperado.')
print('✅ NGROK_TOKEN detectado.')

# ── Limpiar procesos anteriores ───────────────────────────────────────────────
print('\n🧹 Limpiando procesos anteriores...')
try:
    subprocess.run(['pkill', '-f', 'uvicorn'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
except Exception:
    pass

try:
    ngrok.kill()
except Exception:
    pass

time.sleep(2)

# ── Iniciar Uvicorn ───────────────────────────────────────────────────────────
nest_asyncio.apply()

server_process = subprocess.Popen(
    [
        sys.executable, '-m', 'uvicorn', 'main:app',
        '--host', HOST,
        '--port', str(PORT),
        '--timeout-keep-alive', '120'
    ],
    cwd=API_DIR,
    env={**os.environ, 'ANTHROPIC_API_KEY': ANTHROPIC_API_KEY},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = False
start_t = time.time()

print('\n⏳ Esperando arranque (máx 60s)...')

while time.time() - start_t < 60:
    if server_process.poll() is not None:
        print('❌ Uvicorn cerró inesperadamente. Logs:')
        rem = server_process.stdout.read()
        if rem:
            print(rem)
        break

    try:
        line = server_process.stdout.readline()
        if line and line.strip():
            print(f'  📋 {line.rstrip()}')
    except Exception:
        pass

    try:
        r = requests.get(f'http://{HOST}:{PORT}/', timeout=2)
        if r.status_code < 500:
            ready = True
            print(f'\n✅ Servidor listo en {round(time.time() - start_t, 1)}s')
            break
    except Exception:
        pass

    time.sleep(1)

# ── Conectar ngrok en modo FREE (sin domain/subdomain custom) ───────────────
tunnel = None
public_url = None

if ready:
    try:
        # Forma recomendada para registrar el authtoken
        try:
            ngrok.set_auth_token(NGROK_TOKEN)
        except Exception:
            conf.get_default().auth_token = NGROK_TOKEN

        # IMPORTANTE:
        # En plan Free NO usar domain=..., subdomain=... ni custom hostname.
        # Deja que ngrok entregue una URL efímera automática.
        tunnel = ngrok.connect(addr=str(PORT), proto='http')
        public_url = tunnel.public_url

        print('=' * 70)
        print(f'🌍 URL PÚBLICA  : {public_url}')
        print(f'📘 Swagger docs : {public_url}/docs')
        print(f'📗 ReDoc        : {public_url}/redoc')
        print('=' * 70)

        # Verificación externa del túnel
        ok_public = False
        for _ in range(10):
            try:
                rr = requests.get(public_url, timeout=5)
                if rr.status_code < 500:
                    ok_public = True
                    break
            except Exception:
                pass
            time.sleep(1)

        if ok_public:
            print('✅ Backend operativo vía ngrok Free.')
        else:
            print('⚠️ El túnel fue creado, pero la verificación externa no respondió a tiempo.')
            print('   Prueba manualmente la URL pública y /docs.')

    except Exception as e:
        print(f'❌ Error ngrok: {e}')
else:
    print('\n❌ Timeout: el servidor no respondió en 60s.')
    try:
        rem = server_process.stdout.read()
        if rem:
            print(rem)
    except Exception:
        pass

# ── Streaming de logs ─────────────────────────────────────────────────────────
if ready:
    print('\n📜 Streaming de logs (presiona ■ Stop para detener):')
    try:
        while True:
            if server_process.poll() is not None:
                print('⚠️ Servidor detenido.')
                break

            line = server_process.stdout.readline()
            if line:
                print(line.rstrip())

            time.sleep(0.1)

    except KeyboardInterrupt:
        print('🛑 Detenido por el usuario.')
        try:
            server_process.terminate()
        except Exception:
            pass
        try:
            if public_url:
                ngrok.disconnect(public_url)
        except Exception:
            pass